# IEP-1002 Day 2 — Step 3: ChromaDB 벡터 저장 및 스모크 테스트


## Step 2 결과 요약 및 Step 3 목표

**Step 2 결과:**
- 최종 청크: **284개**, 평균 1,669자, 중앙값 1,918자
- fallback 비율: **99.3%** (헤딩 48개 → 원시 청크가 너무 커서 전부 `split_max` 분할)
- 실질적으로 "헤딩 메타데이터가 붙은 고정 길이 청킹"에 가깝지만, 결과물 자체는 RAGAS 평가에 충분

**Step 3 목표:**
1. 라이브러리 설치 및 환경 확인
2. `ipcc_chunks_structural.json` → ChromaDB 컬렉션에 임베딩 저장
3. 골든셋 4유형 각 1개 쿼리로 스모크 테스트 (검색 동작 확인)
4. Day 2 완료 선언 + memory 문서 업데이트용 요약 출력

**완료 기준:** 스모크 테스트 4/4 통과 (각 쿼리에서 관련 청크가 상위 결과에 등장)


### 셀 1 — 라이브러리 설치
- 확인할 것: 설치 오류 없이 완료되는지. `chromadb`와 `sentence-transformers` 버전을 메모해두세요.


In [ ]:
# 셀 1: 라이브러리 설치
!pip install chromadb sentence-transformers -q

import chromadb
import sentence_transformers
print(f"chromadb            : {chromadb.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")


### 셀 2 — 데이터 로드 및 임베딩 모델 초기화

임베딩 모델: `jhgan/ko-sroberta-multitask`
- 한국어 특화 Sentence-BERT 계열 모델
- KLUE, KorNLI, KorSTS 기반 학습 → 한국어 의미 유사도에 강점
- 768차원, 최대 128토큰 (한국어 기준 약 200~300자)

- 확인할 것: 모델이 정상 로드되는지, 청크 284개가 로드됐는지.


In [ ]:
# 셀 2: 데이터 로드 및 임베딩 모델 초기화
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer

OUTPUT_DIR = "/content/drive/MyDrive/IPCC_data/parsed"
DB_DIR     = "/content/drive/MyDrive/IPCC_data/chroma_db"

# 청크 로드
with open(f"{OUTPUT_DIR}/ipcc_chunks_structural.json", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"청크 로드: {len(chunks)}개")
print(f"  첫 청크: {chunks[0]['heading'][:50]}  ({chunks[0]['char_count']}자)")
print(f"  끝 청크: {chunks[-1]['heading'][:50]}  ({chunks[-1]['char_count']}자)")
print()

# 한국어 임베딩 모델 로드
EMBED_MODEL = "jhgan/ko-sroberta-multitask"
print(f"임베딩 모델 로드 중: {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL)
print(f"✓ 모델 로드 완료")
print(f"  임베딩 차원: {embedder.get_sentence_embedding_dimension()}")
print(f"  최대 시퀀스 길이: {embedder.max_seq_length} 토큰")


### 셀 3 — ChromaDB 컬렉션 생성

- 확인할 것: 컬렉션 이름 `ipcc_structural_v1`이 정상 생성됐는지.
- 재실행 시 기존 컬렉션을 자동으로 삭제하고 새로 만듭니다 (멱등성 보장).


In [ ]:
# 셀 3: ChromaDB 클라이언트 및 컬렉션 생성
import chromadb
from chromadb.config import Settings

Path(DB_DIR).mkdir(parents=True, exist_ok=True)

# 영구 저장 클라이언트 (Drive에 저장)
client = chromadb.PersistentClient(path=DB_DIR)

COLLECTION_NAME = "ipcc_structural_v1"

# 재실행 시 기존 컬렉션 삭제 후 재생성
existing = [c.name for c in client.list_collections()]
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)
    print(f"기존 컬렉션 삭제: {COLLECTION_NAME}")

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={
        "hnsw:space"  : "cosine",            # L2 → cosine: 거리값 0~2, 유사도 = 1 - distance
        "description" : "IEP-1002 구조 기반 청킹 (284청크)",
        "embed_model" : "jhgan/ko-sroberta-multitask",
        "chunking"    : "structural_v1",
        "chunk_count" : str(len(chunks)),
    }
)

print(f"✓ 컬렉션 생성: {COLLECTION_NAME}")
print(f"  저장 경로: {DB_DIR}")


### 셀 4 — 임베딩 및 ChromaDB 저장

배치 크기 64로 나눠서 임베딩합니다. Colab 무료 티어 기준 약 3~5분 소요 예상.
- 확인할 것: 진행 상황이 출력되는지, 최종 `collection.count()`가 284인지.


In [ ]:
# 셀 4: 임베딩 일괄 생성 및 ChromaDB 저장
import time

BATCH_SIZE = 64

# ChromaDB에 넣을 데이터 준비
ids       = [f"chunk_{c['chunk_id']:04d}" for c in chunks]
documents = [c["text"] for c in chunks]
metadatas = [
    {
        "chunk_id"   : c["chunk_id"],
        "heading"    : c["heading"][:200],   # ChromaDB metadata 길이 제한 대비
        "pattern"    : c["pattern"],
        "page_start" : c["page_start"],
        "page_end"   : c["page_end"],
        "char_count" : c["char_count"],
        "fallback"   : str(c["fallback"]),  # list → str (ChromaDB는 list 메타 불가)
    }
    for c in chunks
]

# 배치 임베딩 + 저장
total   = len(chunks)
t_start = time.time()

for i in range(0, total, BATCH_SIZE):
    batch_docs  = documents[i:i + BATCH_SIZE]
    batch_ids   = ids[i:i + BATCH_SIZE]
    batch_meta  = metadatas[i:i + BATCH_SIZE]

    embeddings = embedder.encode(
        batch_docs,
        show_progress_bar=False,
        convert_to_numpy=True,
    ).tolist()

    collection.add(
        ids        = batch_ids,
        documents  = batch_docs,
        embeddings = embeddings,
        metadatas  = batch_meta,
    )

    done = min(i + BATCH_SIZE, total)
    elapsed = time.time() - t_start
    print(f"  저장 중... {done:3d}/{total}  ({elapsed:.0f}초)")

elapsed_total = time.time() - t_start
stored_count  = collection.count()

print()
print(f"✓ ChromaDB 저장 완료")
print(f"  저장 청크 수: {stored_count}  (예상: {total})  {'✓' if stored_count == total else '✗ 불일치'}")
print(f"  소요 시간   : {elapsed_total:.1f}초")


### 셀 5 — 스모크 테스트 (골든셋 4유형 각 1개)

각 유형을 대표하는 쿼리 1개씩으로 검색 동작을 확인합니다.  
반환된 청크의 내용이 쿼리와 관련이 있으면 통과입니다.

| 유형 | 기대 동작 |
|---|---|
| 01. 사실 확인 | 구체적 수치가 포함된 청크가 상위에 등장 |
| 02. 비교 | 시나리오 비교 관련 청크가 상위에 등장 |
| 03. 의견/예측 | 미래 전망·리스크 관련 청크가 상위에 등장 |
| 04. 범위 밖 | IPCC 보고서와 무관한 내용 → 유사도 낮은 청크만 반환 |

- 확인할 것: 각 쿼리에서 `score`(유사도)와 반환 청크 내용을 육안으로 확인하세요.


In [ ]:
# 셀 5: 스모크 테스트
import numpy as np

SMOKE_QUERIES = [
    {
        "type"   : "01. 사실 확인",
        "query"  : "2011~2020년 전 지구 지표면 온도는 산업화 이전 대비 몇 도 상승했는가?",
        "expect" : "1.1°C 수치가 포함된 청크",
    },
    {
        "type"   : "02. 비교",
        "query"  : "SSP1-1.9 시나리오와 SSP5-8.5 시나리오의 2100년 온난화 수준 차이",
        "expect" : "시나리오 비교 청크",
    },
    {
        "type"   : "03. 의견/예측",
        "query"  : "1.5도 목표 달성을 위해 2030년까지 온실가스 배출량을 어느 수준으로 줄여야 하는가?",
        "expect" : "2030년 감축 목표 관련 청크",
    },
    {
        "type"   : "04. 범위 밖",
        "query"  : "2024년 한국 대통령 선거 결과는 어떻게 됐는가?",
        "expect" : "유사도 낮음 — IPCC 범위 밖",
    },
]

N_RESULTS  = 3   # 유형별 상위 N개 반환
results_log = []

print("=" * 70)
print("  스모크 테스트 — 골든셋 4유형 각 1개")
print("=" * 70)

for sq in SMOKE_QUERIES:
    q_embed = embedder.encode([sq["query"]], convert_to_numpy=True).tolist()

    res = collection.query(
        query_embeddings = q_embed,
        n_results        = N_RESULTS,
        include          = ["documents", "metadatas", "distances"],
    )

    docs      = res["documents"][0]
    metas     = res["metadatas"][0]
    distances = res["distances"][0]
    # cosine 거리 반환 → 유사도 = 1 - distance  (범위: -1~1, 높을수록 유사)
    scores    = [round(1 - d, 4) for d in distances]

    print(f"\n[{sq['type']}]")
    print(f"  쿼리  : {sq['query']}")
    print(f"  기대  : {sq['expect']}")
    print(f"  {'순위':>2s}  {'유사도':>6s}  {'페이지':>6s}  헤딩 (앞 45자)             내용 미리보기 (앞 80자)")
    print(f"  {'─'*2}  {'─'*6}  {'─'*6}  {'─'*45}  {'─'*80}")

    for rank, (doc, meta, score) in enumerate(zip(docs, metas, scores), 1):
        heading_short = meta["heading"][:45]
        content_short = doc[:80].replace("\n", " ")
        print(f"  {rank:2d}  {score:6.4f}  p.{meta['page_start']:3d}~{meta['page_end']:3d}  "
              f"{heading_short:<45s}  {content_short}")

    results_log.append({
        "type"   : sq["type"],
        "query"  : sq["query"],
        "top1_score"  : scores[0],
        "top1_heading": metas[0]["heading"][:80],
        "top1_page"   : f"p.{metas[0]['page_start']}~{metas[0]['page_end']}",
    })

print()
print("=" * 70)
print("  스모크 테스트 결과 요약")
print("=" * 70)
for r in results_log:
    # 유형 04(범위 밖)는 유사도가 낮아야 정상
    if r["type"].startswith("04"):
        status = "✓ PASS" if r["top1_score"] < 0.5 else "△ 확인 필요 (유사도가 예상보다 높음)"
    else:
        status = "✓ PASS" if r["top1_score"] >= 0.4 else "△ 확인 필요 (유사도가 낮음)"
    print(f"  {r['type']}  score={r['top1_score']:.4f}  {status}")
    print(f"    → {r['top1_heading']}  {r['top1_page']}")


### 셀 6 — 스모크 테스트 로그 저장
- 확인할 것: `smoke_test_structural.txt`가 Drive에 저장됐는지.


In [ ]:
# 셀 6: 스모크 테스트 결과 텍스트 저장
log_path = f"{OUTPUT_DIR}/smoke_test_structural.txt"

lines = ["IEP-1002 스모크 테스트 결과", "=" * 50, ""]
for r in results_log:
    lines.append(f"[{r['type']}]")
    lines.append(f"  쿼리     : {r['query']}")
    lines.append(f"  top1 유사도: {r['top1_score']:.4f}")
    lines.append(f"  top1 헤딩 : {r['top1_heading']}")
    lines.append(f"  top1 페이지: {r['top1_page']}")
    lines.append("")

Path(log_path).write_text("\n".join(lines), encoding="utf-8")
print(f"✓ 저장: {log_path}")


### 셀 7 — Day 2 완료 선언 및 memory 업데이트용 요약

이 출력을 복사해 `memory_260407.md`의 IEP-1002 Day 2 완료 내역에 붙여넣으세요.


In [ ]:
# 셀 7: Day 2 완료 요약 출력
sizes    = [c["char_count"] for c in chunks]
n        = len(sizes)
sorted_s = sorted(sizes)
n_split  = sum(1 for c in chunks if c["fallback"] and "split_max" in c["fallback"])
n_merge  = sum(1 for c in chunks if c["fallback"] and "merge_min" in c["fallback"])
fb_rate  = (n_split + n_merge) / n * 100

pass_count = sum(
    1 for r in results_log
    if (r["type"].startswith("04") and r["top1_score"] < 0.5)
    or (not r["type"].startswith("04") and r["top1_score"] >= 0.4)
)

print(f"""
╔══════════════════════════════════════════════════╗
  IEP-1002 Day 2 완료 요약  (memory 업데이트용)
╚══════════════════════════════════════════════════╝

#### Day 2 완료 (2026-04-08)

**헤딩 전처리 결과**
- Day 1 원본: 150개 → 전처리 후: 68개 → 오탐 제거 후: 48개
- 목차 중복 44개, 러닝헤더 38개, 수동 오탐 3개 제거
- ⚠ 본문 헤딩 보완 시도: 목차 클린 제목으로 재스캔 → 0개 탐지 (문자열 불일치)
  → 원인: 본문 텍스트 내 헤딩이 단독 줄이 아닌 연속 텍스트로 추출됨

**청킹 결과**
| 항목 | 수치 |
| :--- | :--- |
| 최종 청크 수 | {n}개 |
| 평균 청크 크기 | {sum(sizes)//n:,}자 |
| 중앙값 | {sorted_s[n//2]:,}자 |
| 최소 / 최대 | {min(sizes):,}자 / {max(sizes):,}자 |
| fallback 비율 | {fb_rate:.1f}% (split_max {n_split}개, merge_min {n_merge}개) |
| IEP-1001 최적 구간 내 (1000~1200자) | {sum(1 for s in sizes if 1000<=s<=1200)}개 |

⚠ fallback 비율 {fb_rate:.1f}% — 헤딩 48개로 원시 청크가 너무 커서 전부 split_max 분할됨
   실질적으로 "헤딩 메타데이터가 붙은 고정 길이 청킹"에 가까움
   → IEP-1003 의미 기반 청킹 또는 헤딩 탐지 방식 개선 검토 필요

**스모크 테스트**
| 유형 | top1 유사도 | 결과 |
| :--- | :---: | :--- |
""")

for r in results_log:
    status = "PASS" if (
        (r["type"].startswith("04") and r["top1_score"] < 0.5) or
        (not r["type"].startswith("04") and r["top1_score"] >= 0.4)
    ) else "확인 필요"
    print(f"| {r['type']} | {r['top1_score']:.4f} | {status} |")

print(f"""
스모크 테스트: {pass_count}/4 통과

**저장 파일**
- `ipcc_chunks_structural.json`  — 최종 청크 {n}개
- `ipcc_headings_final.json`     — 청킹 기준 헤딩 48개
- `chunk_dist_structural.png`    — 크기 분포 히스토그램
- `chroma_db/`                   — ChromaDB 컬렉션 (`{COLLECTION_NAME}`)
- `smoke_test_structural.txt`    — 스모크 테스트 결과 로그

**Day 3 예정 — RAGAS 4지표 평가**
- Context Recall / Context Precision / Faithfulness / Answer Relevancy
- IEP-1001 CASE 3 (1000/200) Baseline과 비교
""")
